# Merge all files into one

In [3]:
import pandas as pd
from pathlib import Path

# Build file list from 2025-04.csv to 2025-10.csv (inclusive)
files = [Path(f"2025-{month:02d}_R05.csv") for month in range(4, 11)]

# Read and merge
df_merged = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# Export
df_merged.to_csv("2025_full_R05.csv", index=False)

print("Created: 2025_full.csv")

Created: 2025_full.csv


# Check if missing data

In [4]:
# Use existing merged data if available; otherwise read from file
df = df_merged.copy() if "df_merged" in globals() else pd.read_csv("2025_full.csv")

# 1) Missing values check
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame(
    {"missing_count": missing_counts, "missing_percent": missing_pct}
).sort_values("missing_count", ascending=False)

print("=== Missing data by column ===")
print(missing_report[missing_report["missing_count"] > 0] if (missing_counts > 0).any() else "No missing values found.")

# 2) Detect datetime column
candidates = [c for c in ["timestamp", "datetime", "date_time", "time", "date"] if c in df.columns]
dt_col = None

if candidates:
    dt_col = candidates[0]
else:
    best_col, best_valid = None, 0
    for c in df.columns:
        parsed = pd.to_datetime(df[c], errors="coerce")
        valid = parsed.notna().sum()
        if valid > best_valid:
            best_col, best_valid = c, valid
    if best_col is not None and best_valid > 0:
        dt_col = best_col

if dt_col is None:
    print("\nNo datetime-like column found, so 15-minute continuity cannot be checked.")
else:
    df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
    dt = df[dt_col].dropna().sort_values().drop_duplicates()

    if dt.empty:
        print(f"\nColumn '{dt_col}' has no valid datetimes, so 15-minute continuity cannot be checked.")
    else:
        # 3) Check expected 15-minute timeline across full range
        expected = pd.date_range(start=dt.min(), end=dt.max(), freq="15min")
        missing_times = expected.difference(dt)

        print(f"\n=== 15-minute interval check (column: '{dt_col}') ===")
        print(f"Range: {dt.min()} -> {dt.max()}")
        print(f"Expected points: {len(expected)}")
        print(f"Actual unique timestamps: {len(dt)}")
        print(f"Missing 15-minute timestamps: {len(missing_times)}")

        if len(missing_times) > 0:
            missing_df = pd.DataFrame({dt_col: missing_times})
            print("\nFirst 20 missing timestamps:")
            print(missing_df.head(20))
            missing_df.to_csv("missing_15min_timestamps.csv", index=False)
            print("\nSaved missing timestamps to: missing_15min_timestamps.csv")

=== Missing data by column ===
No missing values found.

=== 15-minute interval check (column: 'Unnamed: 0') ===
Range: 2025-04-01 00:00:00+00:00 -> 2025-10-31 22:45:00+00:00
Expected points: 20540
Actual unique timestamps: 20540
Missing 15-minute timestamps: 0


# Full data hourly instead of every 15min

In [7]:
# Read source file
df_hour = pd.read_csv("2025_full_R05.csv")

# Pick datetime column (reuse dt_col from previous cell if available and valid)
if "dt_col" in globals() and dt_col in df_hour.columns:
    time_col = dt_col
else:
    candidates = [c for c in ["timestamp", "datetime", "date_time", "time", "date"] if c in df_hour.columns]
    time_col = candidates[0] if candidates else None
    if time_col is None:
        best_col, best_valid = None, 0
        for c in df_hour.columns:
            parsed = pd.to_datetime(df_hour[c], errors="coerce")
            valid = parsed.notna().sum()
            if valid > best_valid:
                best_col, best_valid = c, valid
        if best_col is not None and best_valid > 0:
            time_col = best_col

if time_col is None:
    raise ValueError("No datetime-like column found in 2025_full.csv")

# Convert time column and keep valid rows
df_hour[time_col] = pd.to_datetime(df_hour[time_col], errors="coerce")
df_hour = df_hour.dropna(subset=[time_col]).sort_values(time_col)

# Resample to 1-hour frequency using mean on numeric columns
hourly = (
    df_hour.set_index(time_col)
           .resample("1h")
           .mean(numeric_only=True)
           .reset_index()
)

# Save result
hourly.to_csv("2025_full_hourly_R05.csv", index=False)
print("Created: 2025_full_hourly_R05.csv")
print(f"Rows: {len(hourly)}")

Created: 2025_full_hourly_R05.csv
Rows: 5135
